In [15]:
from transformers import pipeline

In [16]:
# 1. Sentiment Analysis
sentiment_pipeline = pipeline("sentiment-analysis", device=0)
# Analyze the two sentences
sentence1 = "I've been waiting for a EE471 course my whole life."
sentence2 = "I hate EE471 course"

result1 = sentiment_pipeline(sentence1)
result2 = sentiment_pipeline(sentence2)

print(f"Sentence 1: {sentence1}")
print(f"Sentiment: {result1[0]['label']}, Score: {result1[0]['score']:.4f}")

print(f"\nSentence 2: {sentence2}")
print(f"Sentiment: {result2[0]['label']}, Score: {result2[0]['score']:.4f}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 347.48it/s, Materializing param=pre_classifier.weight]                                  


Sentence 1: I've been waiting for a EE471 course my whole life.
Sentiment: NEGATIVE, Score: 0.9910

Sentence 2: I hate EE471 course
Sentiment: NEGATIVE, Score: 0.9993


In [17]:

classifier = pipeline("zero-shot-classification", device=0)

text = "Berkshire keeps their cash reserves at an extremely high level."
candidate_labels = ["finance", "technology", "healthcare", "business"]
result = classifier(text, candidate_labels)

# Display results
print(f"Text: {text}")
print(f"Predicted Label: {result['labels'][0]}")
print(f"Confidence Score: {result['scores'][0]:.4f}")

# Optional: Show all scores
print("\n--- All Scores ---")
for label, score in zip(result['labels'], result['scores']):
    print(f"{label}: {score:.4f}")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 515/515 [00:01<00:00, 297.59it/s, Materializing param=model.shared.weight]                                   


Text: Berkshire keeps their cash reserves at an extremely high level.
Predicted Label: business
Confidence Score: 0.5123

--- All Scores ---
business: 0.5123
finance: 0.4562
technology: 0.0232
healthcare: 0.0083


In [18]:
generator = pipeline("text-generation")

# Your prompt
prompt = "If I continue to successfully complete all in-class exercises in EE471 course,"

# Generate 2 alternative completions
outputs = generator(
    prompt,
    max_new_tokens=35,        # ~35 words max
    num_return_sequences=2,   # Two different completions
    truncation=True,          # Prevents errors
    temperature=0.85,         # Adds slight creativity
    do_sample=True            # Allows variation between outputs
)

# Print results
print("Prompt:", prompt)
print("\n--- Alternative Completions ---")
for i, output in enumerate(outputs):
    generated_text = output['generated_text']
    print(f"{i+1}: {generated_text}")

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 400.11it/s, Materializing param=transformer.wte.weight]             
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=35) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: If I continue to successfully complete all in-class exercises in EE471 course,

--- Alternative Completions ---
1: If I continue to successfully complete all in-class exercises in EE471 course, I will have a better chance of completing this course in a year which is in line with my current state of performance.

Q: What are your plans for your next
2: If I continue to successfully complete all in-class exercises in EE471 course, I will complete the training under my own permission under my own name, just like I did, if you think that you should pay a fee for these courses, please contact me


In [19]:
# 1. Load the pipeline
unmasker = pipeline("fill-mask", device=0)

# 2. Define the text with a <mask> token where a word should go
text = "The Izmir Institute of Technology is a very <mask> university."

# 3. Run the pipeline
results = unmasker(text)

# 4. Print the top predictions
for res in results:
    print(f"Predicted word: '{res['token_str']}'")
    print(f"Full sentence: {res['sequence']}")
    print(f"Confidence: {res['score']:.4f}\n")

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 106/106 [00:00<00:00, 362.23it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Predicted word: ' prestigious'
Full sentence: The Izmir Institute of Technology is a very prestigious university.
Confidence: 0.3599

Predicted word: ' reputable'
Full sentence: The Izmir Institute of Technology is a very reputable university.
Confidence: 0.0955

Predicted word: ' distinguished'
Full sentence: The Izmir Institute of Technology is a very distinguished university.
Confidence: 0.0922

Predicted word: ' respected'
Full sentence: The Izmir Institute of Technology is a very respected university.
Confidence: 0.0497

Predicted word: ' important'
Full sentence: The Izmir Institute of Technology is a very important university.
Confidence: 0.0332



In [20]:
ner_pipeline = pipeline("ner", aggregation_strategy="simple", device=0)

# 2. Define the text
text = "I am Nate, a research assistant in Izmir Institute of Technology, and currently living and working in beautiful city İzmir in Türkiye."

# 3. Run the pipeline
ner_results = ner_pipeline(text)

# 4. Create variables to hold our specific extractions
subject_name = None
company = None
locations = []

# 5. Loop through the results and sort them by category
for entity in ner_results:
    word = entity['word']
    label = entity['entity_group']
    
    if label == "PER":       # PER stands for Person
        subject_name = word
    elif label == "ORG":     # ORG stands for Organization/Company
        company = word
    elif label == "LOC":     # LOC stands for Location
        locations.append(word)

# Join multiple locations with a comma for a cleaner output
location_string = ", ".join(locations)

# 6. Print the final extracted information
print(f"Original Text: '{text}'\n")
print("Target Extractions:")
print(f"- Subject Name: {subject_name}")
print(f"- Company:      {company}")
print(f"- Location:     {location_string}")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 391/391 [00:01<00:00, 367.77it/s, Materializing param=classifier.weight]                                      
BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Original Text: 'I am Nate, a research assistant in Izmir Institute of Technology, and currently living and working in beautiful city İzmir in Türkiye.'

Target Extractions:
- Subject Name: Nate
- Company:      Izmir Institute of Technology
- Location:     İzmir, Tü, ##iye


In [21]:
# Instead of task="question-answering", just provide the model.
check_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2", 
)

text = "I am Nate, a research assistant in Izmir Institute of Technology, and currently living and working in beautiful city İzmir in Türkiye."

questions = {
    "Subject Name": "What is the name of the person?",
    "Company": "Where does Nate work as a research assistant?",
    "Location": "In which city and country does Nate live?"
}

print("Validating extractions via QA...\n")

for label, question in questions.items():
    # Pass the question and context to the pipeline
    result = check_pipeline(question=question, context=text)
    print(f"- {label} Validation: {result['answer']} (Confidence: {result['score']:.4f})")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 377.46it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Validating extractions via QA...

- Subject Name Validation: Nate (Confidence: 0.8180)
- Company Validation: Izmir Institute of Technology (Confidence: 0.9704)
- Location Validation: İzmir in Türkiye (Confidence: 0.2166)


In [22]:
# 1. Pipeline kurulumu (Text-generation olarak)
summarizer = pipeline("text-generation", model="HuggingFaceTB/SmolLM-135M-Instruct", device=-1)

# 2. Metin tanımı
text = """The 2008 Global Financial Crisis stands as the most severe
economic collapse of the 21st century, often compared to the Great Depression of the
1930s. Triggered by the bursting of the United States housing bubble, its effects rippled
across the globe, leading to the collapse of major financial institutions and a deep
international recession. The crisis began with the subprime mortgage market."""

# 3. Modern Format: Talimat (Prompt) ekliyoruz
prompt = f"Summarize the following text briefly:\n\n{text}\n\nSummary:"

print("Generating summary...")
# max_new_tokens kullanmak daha güvenlidir
summary_results = summarizer(prompt, max_new_tokens=45, do_sample=False)

# 4. Çıktıyı Alma (Anahtar 'generated_text' olmalı)
full_output = summary_results[0]['generated_text']

# Sadece modelin ürettiği kısmı (özeti) almak için prompt'u temizleyelim
generated_summary = full_output.replace(prompt, "").strip()

print(f"\nOriginal Text Length: {len(text.split())} words")
print(f"Summary Length:       {len(generated_summary.split())} words\n")
print("Generated Summary:")
print(f"> {generated_summary}")




Loading weights: 100%|██████████| 272/272 [00:00<00:00, 288.29it/s, Materializing param=model.norm.weight]                              
Both `max_new_tokens` (=45) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating summary...

Original Text Length: 62 words
Summary Length:       31 words

Generated Summary:
> The 2008 Global Financial Crisis was a severe economic downturn that began in 2007 and lasted for several years. It was triggered by the subprime mortgage market bubble, which burst in


In [ ]:
# 1. Using a slightly larger, multi-lingual model (Qwen2.5-0.5B)
# This model is excellent for its size and understands Turkish instructions.
translator_llm = pipeline(
    "text-generation", 
    model="HuggingFaceTB/SmolLM-135M-Instruct", 
    device=0
)

text_to_translate = "The 2008 Global Financial Crisis stands as the most severe economic collapse of the 21st century, often compared to the Great Depression."

# 2. Refined Prompt (using a clearer structure for the model)
prompt = f"English: {text_to_translate}\nSpanish translation:"

print("--- Task 8: Translation Results (Improved v5 Approach) ---\n")

# 3. Generate
# Note: we set max_length=None to remove the warning you saw earlier
result = translator_llm(
    prompt, 
    max_new_tokens=100, 
    do_sample=False,
    max_length=None 
)

# 4. Clean and Print
full_output = result[0]['generated_text']
translated_text = full_output.replace(prompt, "").strip()

print(f"Translated Text:\n{translated_text}")

OSError: Qwen2.5-0.5B is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [11]:
# 1. Initialize the image classification pipeline
# We are using the exact model you requested: google/vit-base-patch16-224
image_classifier = pipeline(
    "image-classification", 
    model="google/vit-base-patch16-224", 
    device=0
)

# 2. Define the image source (can be a local file path or a URL)
image_source = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"

print("--- Task 9: Image Classification Results ---\n")

# 3. Run the pipeline
# The pipeline automatically downloads, resizes, and processes the image
results = image_classifier(image_source)

# 4. Display the top predictions
for result in results:
    print(f"Label: {result['label']:<30} | Confidence: {result['score']:.4f}")

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 473.98it/s, Materializing param=vit.layernorm.weight]                                 


--- Task 9: Image Classification Results ---

Label: lynx, catamount                | Confidence: 0.4335
Label: cougar, puma, catamount, mountain lion, painter, panther, Felis concolor | Confidence: 0.0348
Label: snow leopard, ounce, Panthera uncia | Confidence: 0.0324
Label: Egyptian cat                   | Confidence: 0.0239
Label: tiger cat                      | Confidence: 0.0229


In [14]:
asr_pipeline = pipeline(
    "automatic-speech-recognition", 
    model="openai/whisper-tiny", 
    device=0  # Use 0 for GPU, -1 for CPU
)

# 2. Use a simple URL that we know is public
audio_source = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"

print("--- Task 10: Simplified ASR Results ---")

# 3. No extra settings needed—this model is smart enough by default
result = asr_pipeline(audio_source)

print(f"Transcription: {result['text']}") 

Loading weights: 100%|██████████| 167/167 [00:00<00:00, 497.65it/s, Materializing param=model.encoder.layers.3.self_attn_layer_norm.weight]  


--- Task 10: Simplified ASR Results ---


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

Transcription:  I have a dream, but one day, this nation will rise up, live out the true meaning of its dream.
